In [1]:
import pandas as pd


csv_path = 'figure_data/addt_analysis.csv'
df = pd.read_csv(csv_path)
columns = ["thaw_offset", "freeze_offset"]

for col in columns:
    data = df[col].dropna()
    total = len(data)

    negative_count = (data < 0).sum()
    positive_count = (data > 0).sum()
    zero_count = (data == 0).sum()

    negative_pct = (negative_count / total) * 100
    positive_pct = (positive_count / total) * 100
    zero_pct = (zero_count / total) * 100

    print(f"\nResults for column: {col}")
    print(f"Total number of lake-years: {total}")

    print(f"Negative Values; SAR RFM Earlier than ADD: {negative_count} ({negative_pct:.2f}%)")
    print(f"Positive Values; SAR RFM Later than ADD: {positive_count} ({positive_pct:.2f}%)")
    print(f"Same Day Values: {zero_count} ({zero_pct:.2f}%)")



Results for column: thaw_offset
Total number of lake-years: 150942
Negative Values; SAR RFM Earlier than ADD: 56082 (37.15%)
Positive Values; SAR RFM Later than ADD: 90604 (60.03%)
Same Day Values: 4256 (2.82%)

Results for column: freeze_offset
Total number of lake-years: 150942
Negative Values; SAR RFM Earlier than ADD: 116279 (77.04%)
Positive Values; SAR RFM Later than ADD: 32373 (21.45%)
Same Day Values: 2290 (1.52%)


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import kruskal, mannwhitneyu

# File Processing
addt_path = 'figure_data/addt_analysis.csv'
thermo_path = 'alaska_lakes_ice_phenology_2019-2023.csv'
revisit_path = 'figure_data/fig06_offset_revisit_analysis.csv'

addt_df = pd.read_csv(addt_path)
thermo_df = pd.read_csv(thermo_path)
revisit_df = pd.read_csv(revisit_path)

addt_df = addt_df[['lake_id', 'year', 'thaw_offset', 'freeze_offset']]

thermo_df = thermo_df[['lake_id', 'year',
                       'area_km2',
                       'circularity',
                       'sdi',
                       'convexity',
                       'centroid_lat']]

# Get per-detection gap days from the revisit CSV (deduplicate first)
revisit_df = revisit_df.drop_duplicates(subset=['lake_id', 'year'])
gap_df = revisit_df[['lake_id', 'year', 'ice_off_gap_days', 'ice_on_gap_days']]

thermo_df = thermo_df.rename(columns={'centroid_lat': 'latitude'})
addt_df = addt_df.drop_duplicates(subset=['lake_id', 'year'])
thermo_df = thermo_df.drop_duplicates(subset=['lake_id', 'year'])

merged_df = pd.merge(addt_df, thermo_df, on=['lake_id', 'year'], how='inner')
merged_df = pd.merge(merged_df, gap_df, on=['lake_id', 'year'], how='left')

merged_df["log_area_km2"] = np.log10(merged_df["area_km2"])

print(f"Merged dataset: {len(merged_df)} lake-years")
print(f"Gap data available: {merged_df['ice_off_gap_days'].notna().sum()} thaw, {merged_df['ice_on_gap_days'].notna().sum()} freeze")


# Data
offset_columns = ["thaw_offset", "freeze_offset"]

lake_metrics = [
    "area_km2",
    "log_area_km2",
    "circularity",
    "sdi",
    "convexity",
    "latitude"
]

# Group stats
def compute_stats(df, metric):
    return {
        "mean": df[metric].mean(),
        "median": df[metric].median(),
        "p5": df[metric].quantile(0.05),
        "p25": df[metric].quantile(0.25),
        "p75": df[metric].quantile(0.75),
        "p95": df[metric].quantile(0.95)
    }

# Cliff's Delta (chunked vectorization — exact, no subsampling)
def cliffs_delta(x, y):
    """Compute Cliff's delta using chunked broadcasting for memory safety."""
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    chunk_size = 2000
    greater = 0
    less = 0
    for start in range(0, len(x), chunk_size):
        x_chunk = x[start:start + chunk_size]
        greater += np.sum(x_chunk[:, None] > y[None, :])
        less += np.sum(x_chunk[:, None] < y[None, :])
    return (greater - less) / (len(x) * len(y))


# Helper: run within-column and between-column stats
def run_within_column_stats(data_df, label, offset_cols=None):
    if offset_cols is None:
        offset_cols = offset_columns

    for col in offset_cols:
        print(f"\n\n### Within-column tests for: {col} ({label}) ###")

        data = data_df[['lake_id', 'year', col] + lake_metrics].dropna(subset=[col])

        negative = data[data[col] < 0]
        positive = data[data[col] > 0]
        zero = data[data[col] == 0]

        print(f"  Negative: {len(negative)}, Positive: {len(positive)}, Zero: {len(zero)}")

        groups = {
            "Negative": negative,
            "Positive": positive,
            "Zero": zero
        }
        # Remove empty groups
        groups = {k: v for k, v in groups.items() if len(v) > 0}

        for metric in lake_metrics:
            print(f"\n--- {metric} ---")

            samples = [groups[g][metric].dropna() for g in groups]

            if len(samples) >= 2:
                stat, p = kruskal(*samples)
                print(f"Kruskal-Wallis H = {stat:.4f}, p = {p:.6f}")

                group_names = list(groups.keys())
                for i in range(len(group_names)):
                    for j in range(i+1, len(group_names)):
                        g1 = group_names[i]
                        g2 = group_names[j]
                        x = groups[g1][metric].dropna()
                        y = groups[g2][metric].dropna()
                        if len(x) > 0 and len(y) > 0:
                            u_stat, p_val = mannwhitneyu(x, y, alternative='two-sided')
                            delta = cliffs_delta(x.values, y.values)
                            print(f"{g1} vs {g2}:")
                            print(f"  Mann-Whitney U = {u_stat:.4f}, p = {p_val:.6f}")
                            print(f"  Cliff's Delta = {delta:.4f}")


def run_between_column_stats(data_df, label, use_variable_gap=False):
    print(f"\n\n### Between thaw_offset and freeze_offset comparisons ({label}) ###")

    for metric in lake_metrics:
        print(f"\n--- {metric} ---")

        if use_variable_gap:
            conditions = {
                "Negative": lambda df, c: (
                    (df[c] < 0) &
                    (df[c].abs() > 2 * df["ice_off_gap_days" if c == "thaw_offset" else "ice_on_gap_days"])
                ),
                "Positive": lambda df, c: (
                    (df[c] > 0) &
                    (df[c].abs() > 2 * df["ice_off_gap_days" if c == "thaw_offset" else "ice_on_gap_days"])
                ),
            }
        else:
            conditions = {
                "Negative": lambda df, c: df[c] < 0,
                "Positive": lambda df, c: df[c] > 0,
                "Zero": lambda df, c: df[c] == 0,
            }

        for group_label, condition in conditions.items():
            thaw_group = data_df[condition(data_df, "thaw_offset")][metric].dropna()
            freeze_group = data_df[condition(data_df, "freeze_offset")][metric].dropna()

            if len(thaw_group) > 0 and len(freeze_group) > 0:
                stat, p = mannwhitneyu(thaw_group, freeze_group, alternative='two-sided')
                delta = cliffs_delta(thaw_group.values, freeze_group.values)
                print(f"{group_label} group (thaw n={len(thaw_group)}, freeze n={len(freeze_group)}):")
                print(f"  Mann-Whitney U = {stat:.4f}, p = {p:.6f}")
                print(f"  Cliff's Delta = {delta:.4f}")


# =====================================================================
# Group Stats (descriptive)
# =====================================================================
for col in offset_columns:
    data = merged_df[['lake_id', 'year', col] + lake_metrics].dropna(subset=[col])
    total = len(data)

    negative = data[data[col] < 0]
    positive = data[data[col] > 0]
    zero = data[data[col] == 0]

    print(f"\n==============================")
    print(f"Results for column: {col}")
    print(f"Total number of lake-years: {total}")

    for group_name, group_df in {
        "Negative Values; SAR RFM Earlier than ADD": negative,
        "Positive Values; SAR RFM Later than ADD": positive,
        "Same Day Values": zero
    }.items():
        count = len(group_df)
        pct = (count / total) * 100
        print(f"\n{group_name}: {count} ({pct:.2f}%)")
        for metric in lake_metrics:
            stats = compute_stats(group_df, metric)
            print(f"  {metric}: {stats}")


# =====================================================================
# ALL DATA — Statistical Tests
# =====================================================================
print("\n\n" + "=" * 60)
print("STATISTICAL TESTING RESULTS — ALL DATA")
print("=" * 60)

run_within_column_stats(merged_df, "ALL DATA")
run_between_column_stats(merged_df, "ALL DATA")


# =====================================================================
# FILTERED: beyond 2 revisits (|offset| > 2 * per-detection gap)
# =====================================================================
print("\n\n")
print("#" * 70)
print("# FILTERED SUBSET: |offset| > 2 * per-detection gap (beyond 2 revisits)")
print("#" * 70)

# For within-column tests, filter each column by its own gap
for col in offset_columns:
    gap_col = "ice_off_gap_days" if col == "thaw_offset" else "ice_on_gap_days"
    mask = merged_df[col].abs() > 2 * merged_df[gap_col]
    n_filtered = mask.sum()
    n_total = merged_df[col].notna().sum()
    print(f"\n{col}: {n_filtered} of {n_total} lake-years beyond 2 revisits ({100*n_filtered/n_total:.1f}%)")

# Within-column: thaw filtered
print("\n\n" + "=" * 60)
print("WITHIN-COLUMN: FILTERED (|offset| > 2 * gap)")
print("=" * 60)

for col in offset_columns:
    gap_col = "ice_off_gap_days" if col == "thaw_offset" else "ice_on_gap_days"
    filtered = merged_df[merged_df[col].abs() > 2 * merged_df[gap_col]].copy()
    run_within_column_stats(filtered, f"|{col}| > 2*gap", offset_cols=[col])

# Between-column: filtered
print("\n\n" + "=" * 60)
print("BETWEEN-COLUMN: FILTERED (|offset| > 2 * gap)")
print("=" * 60)
run_between_column_stats(merged_df, "FILTERED >2 revisits", use_variable_gap=True)


Merged dataset: 150942 lake-years
Gap data available: 150942 thaw, 150942 freeze

Results for column: thaw_offset
Total number of lake-years: 150942

Negative Values; SAR RFM Earlier than ADD: 56082 (37.15%)
  area_km2: {'mean': np.float64(0.2716560261000721), 'median': np.float64(0.0629603219695378), 'p5': np.float64(0.0217828137233962), 'p25': np.float64(0.032045665639166), 'p75': np.float64(0.18103444565038426), 'p95': np.float64(1.0688102024491666)}
  log_area_km2: {'mean': np.float64(-1.0535209027970405), 'median': np.float64(-1.200933059727199), 'p5': np.float64(-1.6618860223939145), 'p25': np.float64(-1.4942307035369435), 'p75': np.float64(-0.7422387866416247), 'p95': np.float64(0.02890058951639448)}
  circularity: {'mean': np.float64(0.3575606991252272), 'median': np.float64(0.3759734320795181), 'p5': np.float64(0.1158340975388625), 'p25': np.float64(0.2483251381593465), 'p75': np.float64(0.4714630446924851), 'p95': np.float64(0.5545704270275256)}
  sdi: {'mean': np.float64(1.8

Kruskal-Wallis H = 94.6032, p = 0.000000


Negative vs Positive:
  Mann-Whitney U = 2465098686.0000, p = 0.000000
  Cliff's Delta = -0.0297


Negative vs Zero:
  Mann-Whitney U = 118931363.0000, p = 0.707449
  Cliff's Delta = -0.0034


Positive vs Zero:
  Mann-Whitney U = 197873462.5000, p = 0.003698
  Cliff's Delta = 0.0263

--- log_area_km2 ---
Kruskal-Wallis H = 94.6032, p = 0.000000


...

Positive group (thaw n=18074, freeze n=8115):
  Mann-Whitney U = 83372138.5000, p = 0.000000
  Cliff's Delta = 0.1369

--- convexity ---


Negative group (thaw n=8633, freeze n=30244):
  Mann-Whitney U = 150006874.0000, p = 0.000000
  Cliff's Delta = 0.1491


Positive group (thaw n=18074, freeze n=8115):
  Mann-Whitney U = 69272696.5000, p = 0.000000
  Cliff's Delta = -0.0554

--- latitude ---


Negative group (thaw n=8633, freeze n=30244):
  Mann-Whitney U = 54941694.0000, p = 0.000000
  Cliff's Delta = -0.5791


Positive group (thaw n=18074, freeze n=8115):
  Mann-Whitney U = 125928823.5000, p = 0.000000
  Cliff's Delta = 0.7172
